<div style="
    text-align: center;
    font-size: 28px;
    font-weight: bold;
    ">
    Homework 2 (My Attempt)
</div>

**Name**: Salman Sajjad Siddiqui <br>
**AUS ID**: B00086488 <br>
**Due Date**: July 7, 2024

This is my attempt. There are probably better ways of going about it, but this is what I came up with.

We'll start by defining an `Action` class that stores the resulting destinations along with their corresponding probabilities of being reached.

In [27]:
class Action:
    def __init__(self, name, destinations, probabilities):
        """
        Initialize an Action object with a name, destinations, and probabilities.

        Args:
        - name (str): The name of the action.
        - destinations (list): List of destinations corresponding to each probability.
        - probabilities (list): List of probabilities corresponding to each destination.
        """
        self.name = name
        self.destinations = destinations
        self.probabilities = probabilities
        
    def get_result(self):
        """
        Calculate and return the result of the action including its name and probabilities.

        Returns:
        - list: A list containing the name of the action followed by tuples of (destination, probability).
        """
        results = []
        for destination, probability in zip(self.destinations, self.probabilities):
            results.append((destination, probability))
        results.insert(0, self.name)
        return results

Now, we will define a `State` class to store the possible actions available at each state, along with the state's name and value.

In [28]:
class State:
    def __init__(self, name, reward, actions, value=0):
        """
        Initialize a State object with a name, reward, list of actions, and optional initial value.

        Args:
        - name (str): The name of the state.
        - reward (float): The reward associated with the state.
        - actions (list): List of Action objects available in this state.
        - value (float, optional): Initial value of the state (default is 0).
        """
        self.name = name
        self.reward = reward
        self.actions = actions
        self.value = value

    def get_actions(self):
        """
        Get the results of all actions available in this state.

        Returns:
        - list: A list of results of all actions available in this state.
        """
        if self.actions is None:
            return
        
        moves = []
        for action in self.actions:
            moves.append(action.get_result())
        return moves

We can now proceed to define each state in our problem, although this part is somewhat tedious.

In [29]:
# Define states s1 to s6 with their respective actions
s1 = State(
    name='s1',
    reward=1,
    actions=[
        Action(
            name='a1',
            destinations=['s2', 's3'],
            probabilities=[0.3, 0.7]
        ),
        Action(
            name='a2',
            destinations=['s1', 's4'],
            probabilities=[0.3, 0.7]
        )
    ]
)

s2 = State(
    name='s2',
    reward=1,
    actions=[
        Action(
            name='a1',
            destinations=['s2', 's3'],
            probabilities=[0.3, 0.7]
        ),
        Action(
            name='a2',
            destinations=['s1', 's4'],
            probabilities=[0.3, 0.7]
        )
    ]
)

s3 = State(
    name='s3',
    reward=1,
    actions=[
        Action(
            name='a1',
            destinations=['s1', 's4'],
            probabilities=[0.4, 0.6]
        ),
        Action(
            name='a2',
            destinations=['s2', 's3'],
            probabilities=[0.4, 0.6]
        )
    ]
)

s4 = State(
    name='s4',
    reward=1,
    actions=[
        Action(
            name='a1',
            destinations=['s5', 's6'],
            probabilities=[0.6, 0.4]
        )
    ]
)

s5 = State(
    name='s5',
    reward=0,
    actions=None
)

s6 = State(
    name='s6',
    reward=0,
    actions=None
)

Finally, we can feed these states into a `Problem` class and begin implementing the value iteration/best policy algorithm.

In [30]:
class Problem:
    def __init__(self, states):
        self.states = states

problem = Problem([s1, s2, s3, s4, s5, s6])

In [69]:
def best_policy(problem, iterations = 100, gamma = 1, epsilon = .01):
    actions = {}  # Dictionary to store the best action for each state
    for state in problem.states:
        state.value = 0  # Initialize the value of each state to 0
        actions.update({state.name : None})  # Initialize the action for each state as None

    for iteration in range(iterations):
        delta = 0  
        old_values = {state.name : state.value for state in problem.states}  # Store the old state values

        for state in problem.states:
            if state.actions is None:
                state.value = state.reward  # If the state has no actions, its value is its reward
            else:
                list_of_actions_and_corresponding_values = []  # List to store values for each action
                for action in state.get_actions():
                    list_of_actions_and_corresponding_values.append(
                        (action[0], sum(probability * old_values[destination] for destination, probability in action[1:]))
                    )  # Calculate the value for each action based on transition probabilities

                maximum = max(list_of_actions_and_corresponding_values, key = lambda x : x[1])  # Find the action with the maximum value
                state.value = state.reward + gamma * maximum[1]  # Update the state value
                actions[state.name] = maximum[0]  # Store the best action for the state

        delta = max(state.value - old_values[state.name] for state in problem.states)  # Calculate the maximum change in state values
        if delta <= epsilon * (1 - gamma) / gamma:
            print(f'Values Converged after {iteration} iterations')
            return {state.name : (actions[state.name], state.value) for state in problem.states}  # Return the policy if values have converged
        
    print(f'Not Converged after {iterations} iterations')
    return {state.name : (actions[state.name], state.value) for state in problem.states}  # Return the policy if values did not converge after the maximum iterations

In [77]:
result = best_policy(problem, iterations = 50, gamma = .7, epsilon = .01)

for key, values in result.items():
    print(f"For state {key}, the action is '{values[0]}' and the state value is '{values[1]:.3f}'")

Values Converged after 16 iterations
For state s1, the action is 'a1' and the state value is '3.326'
For state s2, the action is 'a1' and the state value is '3.326'
For state s3, the action is 'a2' and the state value is '3.326'
For state s4, the action is 'a1' and the state value is '1.000'
For state s5, the action is 'None' and the state value is '0.000'
For state s6, the action is 'None' and the state value is '0.000'


On a side note, I asked [Claude](https://claude.ai/) for a cleaner way of writing the algorithm, and this is what I got.

In [67]:
def best_policy(problem, iterations=100, gamma=1, epsilon=0.01):
    """
    Determine the best policy for a given problem using value iteration.

    Args:
    problem: The problem object containing states and their properties.
    iterations: Maximum number of iterations (default: 100).
    gamma: Discount factor (default: 1).
    epsilon: Convergence threshold (default: 0.01).

    Returns:
    A dictionary mapping state names to tuples of (best action, state value).
    """
    # Initialize actions and values
    actions = {state.name: None for state in problem.states}
    for state in problem.states:
        state.value = 0

    for iteration in range(iterations):
        # Store old values for convergence check
        old_values = {state.name: state.value for state in problem.states}
        
        for state in problem.states:
            if state.actions is None:
                # Terminal state
                state.value = state.reward
            else:
                # Calculate values for all actions
                action_values = [
                    (action[0], sum(prob * old_values[dest] for dest, prob in action[1:]))
                    for action in state.get_actions()
                ]
                
                # Choose the best action
                best_action, best_value = max(action_values, key=lambda x: x[1])
                
                # Update state value and best action
                state.value = state.reward + gamma * best_value
                actions[state.name] = best_action

        # Check for convergence
        delta = max(state.value - old_values[state.name] for state in problem.states)
        if delta <= epsilon * (1 - gamma) / gamma:
            print(f'Values converged after {iteration + 1} iterations')
            break
    else:
        print(f'Not converged after {iterations} iterations')

    # Return the best policy and state values
    return {state.name: (actions[state.name], state.value) for state in problem.states}

In [78]:
result = best_policy(problem, iterations = 20, gamma = .7, epsilon = .01)

for key, values in result.items():
    print(f"For state {key}, the action is '{values[0]}' and the state value is '{values[1]:.3f}'")

Values Converged after 16 iterations
For state s1, the action is 'a1' and the state value is '3.326'
For state s2, the action is 'a1' and the state value is '3.326'
For state s3, the action is 'a2' and the state value is '3.326'
For state s4, the action is 'a1' and the state value is '1.000'
For state s5, the action is 'None' and the state value is '0.000'
For state s6, the action is 'None' and the state value is '0.000'


<div style="
    font-size: 24px;
    font-weight: bold;
    ">
    Checking Manual Work
</div>

Before ending, let's quickly check whether our code matches our manual calculations. We'll run it for 1, 2, 3, 4, and 5 iterations. You can refer to the Word document titled "Homework 2 (Report)" to verify that the values are consistent.

**0<sup>th</sup> Iteration**

In [80]:
result = best_policy(problem, iterations = 0, gamma = .7, epsilon = .01)

for key, values in result.items():
    print(f"For state {key}, the action is '{values[0]}' and the state value is '{values[1]:.3f}'")

Not Converged after 0 iterations
For state s1, the action is 'None' and the state value is '0.000'
For state s2, the action is 'None' and the state value is '0.000'
For state s3, the action is 'None' and the state value is '0.000'
For state s4, the action is 'None' and the state value is '0.000'
For state s5, the action is 'None' and the state value is '0.000'
For state s6, the action is 'None' and the state value is '0.000'


**1<sup>st</sup> Iteration**

In [82]:
result = best_policy(problem, iterations = 1, gamma = .7, epsilon = .01)

for key, values in result.items():
    print(f"For state {key}, the action is '{values[0]}' and the state value is '{values[1]:.3f}'")

Not Converged after 1 iterations
For state s1, the action is 'a1' and the state value is '1.000'
For state s2, the action is 'a1' and the state value is '1.000'
For state s3, the action is 'a1' and the state value is '1.000'
For state s4, the action is 'a1' and the state value is '1.000'
For state s5, the action is 'None' and the state value is '0.000'
For state s6, the action is 'None' and the state value is '0.000'


**2<sup>nd</sup> Iteration**

In [87]:
result = best_policy(problem, iterations = 2, gamma = .7, epsilon = .01)

for key, values in result.items():
    print(f"For state {key}, the action is '{values[0]}' and the state value is '{values[1]:.3f}'")

Not Converged after 2 iterations
For state s1, the action is 'a1' and the state value is '1.700'
For state s2, the action is 'a1' and the state value is '1.700'
For state s3, the action is 'a1' and the state value is '1.700'
For state s4, the action is 'a1' and the state value is '1.000'
For state s5, the action is 'None' and the state value is '0.000'
For state s6, the action is 'None' and the state value is '0.000'


**3<sup>rd</sup> Iteration**

In [88]:
result = best_policy(problem, iterations = 3, gamma = .7, epsilon = .01)

for key, values in result.items():
    print(f"For state {key}, the action is '{values[0]}' and the state value is '{values[1]:.3f}'")

Not Converged after 3 iterations
For state s1, the action is 'a1' and the state value is '2.190'
For state s2, the action is 'a1' and the state value is '2.190'
For state s3, the action is 'a2' and the state value is '2.190'
For state s4, the action is 'a1' and the state value is '1.000'
For state s5, the action is 'None' and the state value is '0.000'
For state s6, the action is 'None' and the state value is '0.000'


**4<sup>th</sup> Iteration**

In [89]:
result = best_policy(problem, iterations = 4, gamma = .7, epsilon = .01)

for key, values in result.items():
    print(f"For state {key}, the action is '{values[0]}' and the state value is '{values[1]:.3f}'")

Not Converged after 4 iterations
For state s1, the action is 'a1' and the state value is '2.533'
For state s2, the action is 'a1' and the state value is '2.533'
For state s3, the action is 'a2' and the state value is '2.533'
For state s4, the action is 'a1' and the state value is '1.000'
For state s5, the action is 'None' and the state value is '0.000'
For state s6, the action is 'None' and the state value is '0.000'


**5<sup>th</sup> Iteration**

In [90]:
result = best_policy(problem, iterations = 5, gamma = .7, epsilon = .01)

for key, values in result.items():
    print(f"For state {key}, the action is '{values[0]}' and the state value is '{values[1]:.3f}'")

Not Converged after 5 iterations
For state s1, the action is 'a1' and the state value is '2.773'
For state s2, the action is 'a1' and the state value is '2.773'
For state s3, the action is 'a2' and the state value is '2.773'
For state s4, the action is 'a1' and the state value is '1.000'
For state s5, the action is 'None' and the state value is '0.000'
For state s6, the action is 'None' and the state value is '0.000'
